# Assignment 4 – LSTM Text Generation on Epicurious Recipes

Based on the textbook code: *Generative Deep Learning (2nd ed.)*, `notebooks/05_autoregressive/01_lstm/lstm.ipynb`.

Steps: (1) clean & tokenize the recipe text, (2) train an LSTM to predict the next word, (3) generate long text by repeatedly feeding a sequence, sampling the next word, appending it, and repeating. Output is shown for **two temperatures: 0.5 and 0.8** (not 1.0 or 0.2).

In [1]:
import numpy as np
import json, re, string
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, losses

tf.keras.utils.set_random_seed(42)

## 0. Parameters

In [2]:
VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 100
N_UNITS = 128
SEED = 42
BATCH_SIZE = 32
EPOCHS = 25
DATA_PATH = "../data/epirecipes/full_format_recipes.json"
TEMPERATURES = [0.5, 0.8]

## 1. Load the data

In [3]:
with open(DATA_PATH) as f:
    recipe_data = json.load(f)

# Keep recipes that have a title and directions
filtered_data = [
    "Recipe for " + x["title"] + " | " + " ".join(x["directions"])
    for x in recipe_data
    if "title" in x and x["title"] is not None
    and "directions" in x and x["directions"] is not None
]
print(f"{len(filtered_data)} recipes loaded")
print(filtered_data[9])

20111 recipes loaded
Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas  | Chop enough parsley leaves to measure 1 tablespoon; reserve. Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan, covered, 5 minutes. Meanwhile, sprinkle gelatin over water in a medium bowl and let soften 1 minute. Strain broth through a fine-mesh sieve into bowl with gelatin and stir to dissolve. Season with salt and pepper. Set bowl in an ice bath and cool to room temperature, stirring. Toss ham with reserved parsley and divide among jars. Pour gelatin on top and chill until set, at least 1 hour. Whisk together mayonnaise, mustard, vinegar, 1/4 teaspoon salt, and 1/4 teaspoon pepper in a large bowl. Stir in celery, cornichons, and potatoes. Pulse peas with marjoram, oil, 1/2 teaspoon pepper, and 1/4 teaspoon salt in a food processor to a coarse mash. Layer peas, then potato salad, over ham.


## 2. Clean and tokenize the data

In [4]:
# Pad punctuation so each punctuation mark is treated as a separate 'word'
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s

text_data = [pad_punctuation(x) for x in filtered_data]
print(text_data[9])

Recipe for Ham Persillade with Mustard Potato Salad and Mashed Peas | Chop enough parsley leaves to measure 1 tablespoon ; reserve . Chop remaining leaves and stems and simmer with broth and garlic in a small saucepan , covered , 5 minutes . Meanwhile , sprinkle gelatin over water in a medium bowl and let soften 1 minute . Strain broth through a fine - mesh sieve into bowl with gelatin and stir to dissolve . Season with salt and pepper . Set bowl in an ice bath and cool to room temperature , stirring . Toss ham with reserved parsley and divide among jars . Pour gelatin on top and chill until set , at least 1 hour . Whisk together mayonnaise , mustard , vinegar , 1 / 4 teaspoon salt , and 1 / 4 teaspoon pepper in a large bowl . Stir in celery , cornichons , and potatoes . Pulse peas with marjoram , oil , 1 / 2 teaspoon pepper , and 1 / 4 teaspoon salt in a food processor to a coarse mash . Layer peas , then potato salad , over ham . 


In [5]:
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000, seed=SEED)
)

vectorize_layer = layers.TextVectorization(
    standardize="lower",
    max_tokens=VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_LEN + 1,
)
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()

for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")
print(vectorize_layer(text_data[9]).numpy())

0: 
1: [UNK]
2: .
3: ,
4: and
5: to
6: in
7: the
8: with
9: a
[  26   16  557    1    8  298  335  189    4 1054  494   27  332  228
  235  262    5  594   11  133   22  311    2  332   45  262    4  671
    4   70    8  171    4   81    6    9   65   80    3  121    3   59
   12    2  299    3   88  650   20   39    6    9   29   21    4   67
  529   11  164    2  320  171  102    9  374   13  643  306   25   21
    8  650    4   42    5  931    2   63    8   24    4   33    2  114
   21    6  178  181 1245    4   60    5  140  112    3   48    2  117
  557    8  285  235    4  200  292  980    2  107  650   28   72    4
  108   10  114    3   57  204   11  172    2   73  110  482    3  298
    3  190    3   11   23   32  142   24    3    4   11   23   32  142
   33    6    9   30   21    2   42    6  353    3 3224    3    4  150
    2  437  494    8 1281    3   37    3   11   23   15  142   33    3
    4   11   23   32  142   24    6    9  291  188    5    9  412  572
    2  230  494

## 3. Create the training set
Inputs are the token sequence; targets are the same sequence shifted one word to the right.

In [6]:
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)
    tokenized = vectorize_layer(text)
    return tokenized[:, :-1], tokenized[:, 1:]

train_ds = text_ds.map(prepare_inputs).prefetch(tf.data.AUTOTUNE)

## 4. Build the LSTM

In [7]:
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(VOCAB_SIZE, EMBEDDING_DIM)(inputs)
x = layers.LSTM(N_UNITS, return_sequences=True)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
lstm = models.Model(inputs, outputs)
lstm.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 100)      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 128)      │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 10000)    │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,407,248 (9.18 MB)

 Trainable params: 2,407,248 (9.18 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Train the LSTM

In [8]:
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word):
        super().__init__()
        self.index_to_word = index_to_word
        self.word_to_index = {w: i for i, w in enumerate(index_to_word)}

    def sample_from(self, probs, temperature):
        # Rescale the distribution by temperature, then sample
        probs = probs.astype("float64") ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature, verbose=True):
        start_tokens = [self.word_to_index.get(x, 1) for x in start_prompt.split()]
        sample_token = None
        info = []
        # Feed the sequence, predict the next word, append it, repeat
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y = self.model.predict(x, verbose=0)
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append({"prompt": start_prompt, "word_probs": probs})
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        if verbose:
            print(f"\ngenerated text (temperature={temperature}):\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        self.generate("recipe for", max_tokens=100, temperature=0.8)

lstm.compile("adam", losses.SparseCategoricalCrossentropy())
text_generator = TextGenerator(vocab)
checkpoint_cb = callbacks.ModelCheckpoint(
    filepath="./checkpoint/lstm.weights.h5", save_weights_only=True, save_freq="epoch"
)

In [9]:
history = lstm.fit(train_ds, epochs=EPOCHS, callbacks=[checkpoint_cb, text_generator])
lstm.save("./models/lstm.keras")

Epoch 1/25
629/629 - 87s 138ms/step - loss: 5.0047

generated text (temperature=0.8):
recipe for beef poached white chocolate with remaining the wild pasta | and mashed lime a bowl . cook until made 2 - inch - inch - cooking - inch from in skillet . 2 custard carrots and pepper . if made until smooth . strain with paper , about 4 minutes occasionally and broil potatoes , turning , to a mixture , until necessary . salt and add potatoes . dill with 1 , lemon 10 minutes , and pepper . peel with salt . add 2 cup remaining milk , and heavy pot , stirring to 5 minutes

Epoch 2/25
629/629 - 89s 141ms/step - loss: 3.0025

generated text (temperature=0.8):
recipe for [UNK] with the zucchini and finely pomegranate sauce | combine the salt , and chiles in large saucepan over medium bowl . whisk potatoes and add 1 / 4 teaspoon the lemon juice , and pepper ; cook until browned , slightly frozen , about 3 minutes . stir in the sugar ; cook , stirring frequently , stirring occasionally , stirring occ

## 6. Generate long text with two temperatures (0.5 and 0.8)

Lower temperature makes the distribution sharper (safer, more repetitive text); higher temperature flattens it (more varied, riskier text).

In [10]:
def print_probs(info, vocab, top_k=5):
    for i in info:
        print(f"\nPROMPT: {i['prompt']}")
        p = i["word_probs"]
        for idx in np.argsort(p)[::-1][:top_k]:
            print(f"{vocab[idx]}:   \t{np.round(100 * p[idx], 2)}%")
        print("--------")

In [11]:
prompts = [
    "recipe for roasted vegetables |",
    "recipe for chocolate ice cream |",
    "recipe for grilled salmon with lemon |",
]
for t in TEMPERATURES:
    print("=" * 80)
    print(f"TEMPERATURE = {t}")
    print("=" * 80)
    for p in prompts:
        text_generator.generate(p, max_tokens=150, temperature=t)

TEMPERATURE = 0.5



generated text (temperature=0.5):
recipe for roasted vegetables | preheat oven to 400°f . heat oil in heavy large skillet over medium - high heat . add onions ; sauté until golden , about 8 minutes . add garlic and sauté until brown and tender , about 8 minutes . add garlic ; sauté until chicken is cooked through , about 5 minutes . add 1 / 2 cup broth and boil until reduced to 3 tablespoons , about 5 minutes . add wine , broth and bay leaf . bring to boil . reduce heat to medium - low and simmer until potatoes are tender , about 20 minutes . transfer to bowl . add pasta ; toss to coat . cover and chill until cold , about 30 minutes . ( can be made 1 day ahead . cover and keep refrigerated . ) 




generated text (temperature=0.5):
recipe for chocolate ice cream | pour 1 cup cream into a medium saucepan and let steep , covered , 10 minutes . pour off all but 3 tablespoons of juice and discard liquid from pod . bring sugar , water , sugar , and salt to a boil in a 1 - quart saucepan , stirring until sugar is dissolved . in a bowl whisk together sugar , milk , lemon juice , and salt and pepper to taste , stirring until combined well . add cream mixture and beat until smooth . whisk in sour cream , 1 / 4 cup water , and remaining 1 / 2 teaspoon salt and 1 / 2 teaspoon vanilla . in a bowl whisk together yolks , sugar , and vanilla and beat until combined well . gradually whisk in eggs , 1 at a time , beating well after each




generated text (temperature=0.5):
recipe for grilled salmon with lemon | in a bowl combine the onion , the lemon juice , the gingerroot , the vinegar , the olive oil , the sugar , the mustard , the remaining 1 / 4 teaspoon salt , and the cayenne and whisk in the remaining 1 / 2 cup sugar , the tabasco , the cayenne , the oil , the mustard , the salt , and the pepper and the lemon juice and whisk in the remaining ingredients . the olive oil keeps , covered , and refrigerated for up to 2 weeks . 

TEMPERATURE = 0.8



generated text (temperature=0.8):
recipe for roasted vegetables | preheat oven to 450°f . combine all of oil , red peppers , oregano , salt , pepper , and 1 / 2 teaspoon salt in a plastic bag ; let stand , covered tightly , until firm , 10 minutes . turn lamb , working in batches , fry bread until golden brown , about 2 minutes per side . transfer [UNK] to a paper towel–lined baking sheet . ( you can cover the dish clean , use some of your hands or a food processor ) . chop . heat oil in heavy large pot over medium - high heat . add spices and sauté until brown , about 1 minute . add olives , thyme , rosemary , salt and pepper . bring to boil , stirring often . reduce heat to medium and simmer until reduced to 1 1 / 2




generated text (temperature=0.8):
recipe for chocolate ice cream | bring cream , milk , and cream in a small saucepan to blend , swirling pan . reduce heat and simmer , stirring constantly , until smooth . remove from heat and add nutmeg , mint , and remaining 1 / 2 cup milk . add preserves ; bring just to coat . let stand until sugar dissolves and mixture is thickened , about 15 minutes . return butter to 350°f . add 1 1 / 2 cups maple syrup and simmer until mixture is thickened and reduced to 1 1 / 2 to 3 cups , 8 to 12 minutes . sift powdered cornstarch into a large bowl . whisk almond mixture into yolk mixture until smooth ; set aside . blend flour , flour , baking powder , and 1 / 4 cup sugar in a food processor . add 1 /




generated text (temperature=0.8):
recipe for grilled salmon with lemon | pour a small wooden spoons as the soy sauce into a glaze ( it will bubble up ) . sprinkle the chicken mixture with the salt . 



### Next-word probabilities at each temperature

In [12]:
for t in TEMPERATURES:
    print("=" * 80, f"\nTEMPERATURE = {t}")
    info = text_generator.generate("recipe for roasted vegetables | chop 1 /", max_tokens=10, temperature=t)
    print_probs(info, vocab)

TEMPERATURE = 0.5

generated text (temperature=0.5):
recipe for roasted vegetables | chop 1 / 2 cups


PROMPT: recipe for roasted vegetables | chop 1 /
2:   	64.51%
4:   	33.01%
3:   	1.5%
8:   	0.93%
spice:   	0.02%
--------

PROMPT: recipe for roasted vegetables | chop 1 / 2
cup:   	78.51%
cups:   	4.28%
teaspoon:   	3.99%
tsp:   	2.45%
garlic:   	1.6%
--------
TEMPERATURE = 0.8

generated text (temperature=0.8):
recipe for roasted vegetables | chop 1 / 2 pounds


PROMPT: recipe for roasted vegetables | chop 1 /
2:   	53.9%
4:   	35.46%
3:   	5.14%
8:   	3.81%
spice:   	0.36%
--------

PROMPT: recipe for roasted vegetables | chop 1 / 2
cup:   	40.43%
cups:   	6.56%
teaspoon:   	6.28%
tsp:   	4.63%
garlic:   	3.54%
--------


In [13]:
for t in TEMPERATURES:
    print("=" * 80, f"\nTEMPERATURE = {t}")
    info = text_generator.generate("recipe for chocolate ice cream |", max_tokens=7, temperature=t)
    print_probs(info, vocab)

TEMPERATURE = 0.5

generated text (temperature=0.5):
recipe for chocolate ice cream | preheat


PROMPT: recipe for chocolate ice cream |
combine:   	27.14%
in:   	20.98%
preheat:   	14.31%
whisk:   	11.49%
bring:   	8.53%
--------
TEMPERATURE = 0.8

generated text (temperature=0.8):
recipe for chocolate ice cream | bring


PROMPT: recipe for chocolate ice cream |
combine:   	16.24%
in:   	13.82%
preheat:   	10.88%
whisk:   	9.49%
bring:   	7.88%
--------


## 7. Discussion: comparing the two temperatures

The model trained for 25 epochs and the loss went down from about 5.00 after the first epoch to 1.52 at the end. You can actually see this in the samples printed during training: after epoch 1 the text is mostly random words thrown together, but by the last few epochs it looks a lot like a real Epicurious recipe, with measurements, cooking times and temperatures. For the final outputs I used temperatures of 0.5 and 0.8. Temperature controls how "safe" the model is when it picks the next word: a lower value makes it stick to the words it thinks are most likely, and a higher value gives the less likely words more of a chance. You can see this in the probability tables. After "chop 1 / 2", the word "cup" had about 78.5% probability at 0.5, but only about 40.4% at 0.8, so the other options get spread out more.

At 0.5 the recipes were easier to read and sounded more like real directions, for example "preheat oven to 400°f . heat oil in heavy large skillet over medium - high heat . add onions ; sauté until golden". The problem was that it got repetitive. In the roasted vegetables recipe it said "add garlic" twice in a row, and the grilled salmon recipe turned into one long list that kept repeating the lemon juice, mustard and cayenne. It also still made some strange choices, like adding "salt and pepper to taste" to the chocolate ice cream. At 0.8 the text was more varied and creative, with more ingredients and steps mixed in, but it made a lot less sense overall. The roasted vegetables recipe somehow ended up turning lamb and frying bread, the ice cream recipe said "return butter to 350°f", and an unknown-word [UNK] token even showed up once. The grilled salmon one at 0.8 was also really short and didn't make much sense. Overall I think 0.5 gave the more believable recipes and 0.8 was more interesting but less reliable, so something in between would probably be a good balance. What I found most interesting is that the model doesn't really understand cooking at all. It is just predicting one word at a time based on patterns it learned from the training data, and the temperature decides how closely it follows those patterns.